In [1]:
from torch_geometric.datasets import MoleculeNet
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import clear_output
from scipy.spatial import cKDTree
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import torch.nn as nn
import torch_geometric.nn as geom_nn
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler,RobustScaler

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

import torch
from torch.nn import Linear
import torch.nn.functional as F 
from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from torch_geometric.nn.dense import DenseGCNConv
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn import HypergraphConv

Matplotlib created a temporary cache directory at /localscratch-ssd/309943/matplotlib-09veacxh because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
location = "../extrapolation/rho2_30percent_Re100/80_20_time_split/"

train_input = pd.read_csv(location+"train_input")
test_input = pd.read_csv(location+"test_input")

train_input_global = pd.read_csv(location+"train_input_scalar")
test_input_global = pd.read_csv(location+"test_input_scalar")

train_output = pd.read_csv(location+"train_output")
test_output = pd.read_csv(location+"test_output")

### Removing first columns ###
train_input = train_input.drop(['Unnamed: 0',"time"], axis=1)
train_input_global = train_input_global.drop('Unnamed: 0', axis=1)
train_output = train_output.drop('Unnamed: 0', axis=1)
    
test_input = test_input.drop(['Unnamed: 0',"time"], axis=1)
test_input_global = test_input_global.drop('Unnamed: 0', axis=1)
test_output = test_output.drop('Unnamed: 0', axis=1)

# ### drop any columns if needed ###
columns_to_drop = train_input.filter(regex='^(vpx_|vpy_|vpz_)').columns
train_input = train_input.drop(columns=columns_to_drop)

# columns_to_drop = test_input.filter(regex='^(vpx_|vpy_|vpz_)').columns
test_input = test_input.drop(columns=columns_to_drop)

### reshaping to a nodal format ###
train_input = train_input.values.reshape(train_input.shape[0],16,4)
test_input = test_input.values.reshape(test_input.shape[0],16,4)

### global from dataframe to numpy ###
train_input_global = torch.tensor(train_input_global.values).float().clone().detach()
test_input_global = torch.tensor(test_input_global.values).float().clone().detach()

### outputs from dataframe to torch.tensor ###
train_output = train_output.values
test_output = test_output.values

### for rescaling back to original space ###
drag_min = pd.read_csv(location+"test_output_unscaled")["Drag"].values.min()
drag_max = pd.read_csv(location+"test_output_unscaled")["Drag"].values.max()

print("size of train and test datasets : ",train_input.shape,test_input.shape)

size of train and test datasets :  (36806, 16, 4) (9164, 16, 4)


In [3]:
### load the scaler ###
import joblib
scaler = RobustScaler()
scaler = joblib.load(location + "scaler.save") 

In [4]:
### edge index for basic connections ###

### for bi-directional message passing ###
edge_index = torch.tensor([[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                           [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]])

train_combined = list()
test_combined = list()

### Stacking up train data ###
for i in range(len(train_input)):

    ### setting inputs ###
    x = torch.tensor(train_input[i]).float().clone().detach()
    
    ### adding drag force as y ###
    y = torch.tensor(train_output[i]).float().clone().detach()
    
    # all_data_graph_struct.append(Data(x=x , edge_index=torch.tensor(mirror_edge_index(edge_index)) , y=y))
    train_combined.append(Data(x=x.clone().detach() , edge_index=edge_index.clone().detach() , y=y[:,None].clone().detach()))

### Stacking up test data ###
for i in range(len(test_input)):

    ### setting inputs ###
    x = torch.tensor(test_input[i]).float().clone().detach()
    
    ### adding drag force as y ###
    y = torch.tensor(test_output[i]).float().clone().detach()
    
    # all_data_graph_struct.append(Data(x=x , edge_index=torch.tensor(mirror_edge_index(edge_index)) , y=y))
    test_combined.append(Data(x=x.clone().detach() , edge_index=edge_index.clone().detach() , y=y[:,None].clone().detach()))

In [5]:
def gen_edge_index_single(num_nodes):
    """
    Create an edge index that ensures two-way message passing from the first node
    (node 0) to every other node, and vice versa, with edges arranged as:
    [[0, 0, 0, 1, 2, 3],
     [1, 2, 3, 0, 0, 0]].

    Parameters:
    num_nodes (int): The total number of nodes in the graph.

    Returns:
    torch.Tensor: The edge index tensor of shape (2, num_edges).
    """
    # Ensure that there are more than one node
    if num_nodes < 2:
        raise ValueError("There must be at least 2 nodes.")

    # Create edges from node 0 to all other nodes
    src = [0] * (num_nodes - 1)  # 0, 0, 0...
    tgt = list(range(1, num_nodes))  # 1, 2, 3...

    # Add reverse edges from all other nodes back to node 0
    src += list(range(1, num_nodes))  # 1, 2, 3...
    tgt += [0] * (num_nodes - 1)  # 0, 0, 0...

    # Combine the edges into a single edge_index tensor
    edge_index = torch.tensor([src, tgt], dtype=torch.long)
    
    return edge_index

In [6]:
def gen_edge_index_multiple(edge_index_single,num_nodes_per_graph,num_graphs):
    
    return torch.hstack([edge_index_single+num_nodes_per_graph*i for i in range(num_graphs)])

In [7]:
def batch_indexing_function(batch_index,num_nodes):
    
    _,indices = np.unique(batch_index.detach().cpu().numpy(),return_index=True)
    
    ### nodes to extract ###
    nodes_to_extract = [np.arange(num_nodes)+indices[i] for i in range(len(indices))]
    
    ### batch index for final pooling ###
    batch_index_modified = [ nodes_to_extract[i]*0+i for i in range(len(nodes_to_extract))]
    
    return torch.tensor(np.hstack(nodes_to_extract)),torch.tensor(np.hstack(batch_index_modified))

In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool as gap
from torch.nn import Linear

class GCN(torch.nn.Module):
    def __init__(self, embedding_size=200, num_nodes=16, num_features=4):
        
        # Init parent
        super(GCN, self).__init__()
        self.num_nodes = num_nodes

        # GCN layers
        self.initial_conv = GCNConv(num_features, embedding_size)
        self.conv1 = GCNConv(embedding_size, embedding_size)
        self.conv2 = GCNConv(embedding_size, embedding_size)
        self.conv3 = GCNConv(embedding_size, embedding_size)

        # Output layers
        self.linear_1 = Linear(embedding_size + 4, 128)
        self.linear_2 = Linear(128,1)

    def forward(self,x,x_scalar,batch,num_nodes_to_use):
        
        ### Defining edge connections and batch indices based on 
        
        ### Define single edge connection ###
        edge_index_single = gen_edge_index_single(num_nodes_to_use)
        
        ### Expand edge connections across multiple graphs ###
        edge_index = gen_edge_index_multiple(edge_index_single=edge_index_single,
                                             num_nodes_per_graph=self.num_nodes,
                                             num_graphs=batch.batch.max().item() + 1).to(device)
        
        ### Define batch indexing for node extraction and global pooling ###
        nodes_to_extract,batch_index_modified = batch_indexing_function(batch.batch,num_nodes_to_use)
        
        # First Conv layer
        hidden = self.initial_conv(x, edge_index)
        hidden = F.leaky_relu(hidden)
        
        # Other Conv layers
        hidden = self.conv1(hidden, edge_index)
        hidden = F.leaky_relu(hidden)
        
        hidden = self.conv2(hidden, edge_index)
        hidden = F.leaky_relu(hidden)
        
        hidden = self.conv3(hidden, edge_index)
        hidden = F.leaky_relu(hidden)
        
        ### Extracting only relavant nodes ###
        hidden = hidden[nodes_to_extract]
        
        # Global average pooling
        hidden = gap(hidden, batch_index_modified.to(device))
        
        # Concatenate with scalar features
        hidden = torch.cat((hidden, x_scalar), axis=1)
        
        # print("Edge Index",edge_index,"\n")
        # print("Nodes to extract",nodes_to_extract,"\n")
        # print("batch_index_modified",batch_index_modified)
        
        # Final MLP layers
        out = self.linear_1(hidden)
        out = F.leaky_relu(out)
        out = self.linear_2(out)

        return out

# Create the model
model = GCN()
print(model)
print("Number of parameters: ", sum(p.numel() for p in model.parameters()))

GCN(
  (initial_conv): GCNConv(4, 200)
  (conv1): GCNConv(200, 200)
  (conv2): GCNConv(200, 200)
  (conv3): GCNConv(200, 200)
  (linear_1): Linear(in_features=204, out_features=128, bias=True)
  (linear_2): Linear(in_features=128, out_features=1, bias=True)
)
Number of parameters:  147969


In [16]:
# Wrap data in a data loader
NUM_GRAPHS_PER_BATCH = 16
N=-1
train_loader = DataLoader(list(zip(train_combined[0:N],train_input_global[0:N])),
                    batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)

test_loader = DataLoader(list(zip(test_combined,test_input_global)),
                    batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)

In [17]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)  

for batch,inputs_global in train_loader:
    
    batch.to(device)
    inputs_global = inputs_global.float().cuda()

    optimizer.zero_grad()
    
    drag_predictions = model(batch.x.float(),inputs_global,batch,num_nodes_to_use=16)
    
    break

# Run Training

In [161]:
from torch_geometric.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import warnings
warnings.filterwarnings("ignore")

# Root mean squared error
# loss_fn = torch.nn.MSELoss()
loss_fn = torch.nn.HuberLoss(delta=0.25)
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)  

# Use GPU for training
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

### lr scheduler ###
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.75, patience=5, verbose=True)

epoch_loss_train = list()
epoch_loss_val = list()
lr_list = list()

save_loc = "temp_results/"

In [162]:
def train_model_in_stages(model, train_loader, test_loader, save_loc, device, optimizer, loss_fn, num_node_stages=[5, 10, 15], max_epochs=50, early_stopping_patience=5):
    
    # Loop over the stages of num_nodes_to_use
    
    for num_nodes_to_use in num_node_stages:
        
        print(f"Training model with num_nodes_to_use={num_nodes_to_use}")
        
        # Reset learning rate for each stage
        optimizer.param_groups[0]['lr'] = 0.00075
        
        # Initialize scheduler and early stopping variables
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.75, patience=5, verbose=True)
        best_val_loss = float('inf')
        early_stop_counter = 0
        best_model_state = None
        
        epoch_loss_train = []
        epoch_loss_val = []
        lr_list = []
        
        for epoch in range(max_epochs):
            print(f'Starting Epoch {epoch+1} with num_nodes_to_use={num_nodes_to_use}')
            
            loss_train = []
            loss_val = []
            
            # Training loop
            model.train()
            for batch, inputs_global in train_loader:
                batch.to(device)
                inputs_global = inputs_global.float().cuda()

                optimizer.zero_grad()

                pred = model(batch.x.float(), inputs_global, batch, num_nodes_to_use=num_nodes_to_use)
                loss = loss_fn(pred, batch.y)
                loss.backward()

                optimizer.step()

                loss_train.append(loss.item())
            
            # Validation loop
            model.eval()
            with torch.no_grad():
                for batch, inputs_global in test_loader:
                    batch.to(device)
                    inputs_global = inputs_global.float().cuda()

                    pred = model(batch.x.float(), inputs_global, batch, num_nodes_to_use=num_nodes_to_use)
                    loss = loss_fn(pred, batch.y)
                    loss_val.append(loss.item())

            print(f'Epoch {epoch+1}: Training loss = {np.mean(loss_train)}, Validation loss = {np.mean(loss_val)}')
            
            # Store epoch losses
            epoch_loss_train.append(np.mean(loss_train))
            epoch_loss_val.append(np.mean(loss_val))
            
            # Save the best model if validation loss improves
            if epoch_loss_val[-1] < best_val_loss:
                best_val_loss = epoch_loss_val[-1]
                best_model_state = model.state_dict().copy()  # Save best model state
                torch.save(best_model_state, save_loc + f'best_model_num_nodes_{num_nodes_to_use}.pt')
                early_stop_counter = 0
            else:
                early_stop_counter += 1
            
            # Learning rate scheduling
            lr_list.append(optimizer.param_groups[0]['lr'])
            scheduler.step(epoch_loss_val[-1])

            # Early stopping condition
            if early_stop_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1} with num_nodes_to_use={num_nodes_to_use}")
                break
            
            # Save model every 25 epochs
            if (epoch + 1) % 25 == 0:
                torch.save(model.state_dict(), save_loc + f'model_epoch_{epoch+1}_num_nodes_{num_nodes_to_use}.pt')

            # Save loss history to disk
            np.save(save_loc + "epoch_loss_train_"+str(num_nodes_to_use)+"_nodes.npy", epoch_loss_train)
            np.save(save_loc + "epoch_loss_val_"+str(num_nodes_to_use)+"_nodes.npy", epoch_loss_val)
        
        print(f"Training completed for num_nodes_to_use={num_nodes_to_use}. Best validation loss: {best_val_loss}")
        
    print("Training across all stages has completed.")
    return best_model_state  # Optionally return the best model state from the final stage

In [163]:
# Train the model #

# Assuming the required objects like model, train_loader, test_loader, etc., are already defined
loss_fn = torch.nn.HuberLoss(delta=0.25)
optimizer = torch.optim.Adam(model.parameters(),lr=0.00075)  

# Use GPU for training
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Train model in stages with sequential num_nodes_to_use
best_model_state = train_model_in_stages(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    save_loc="temp_results/",  # Directory where model and losses are saved
    device=device,
    optimizer=optimizer,
    loss_fn=loss_fn,
    num_node_stages=[5, 10, 15],  # Sequential training with different num_nodes_to_use
    max_epochs=100,  # Max epochs per stage
    early_stopping_patience=50  # Early stopping after 5 epochs of no improvement
)

# Optionally load the best model state for further use
# model.load_state_dict(best_model_state)

Training model with num_nodes_to_use=5
Starting Epoch 1 with num_nodes_to_use=5
Epoch 1: Training loss = 0.0993060141641208, Validation loss = 0.11088538847656713
Starting Epoch 2 with num_nodes_to_use=5
Epoch 2: Training loss = 0.09152818806036986, Validation loss = 0.10649863464964761
Starting Epoch 3 with num_nodes_to_use=5
Epoch 3: Training loss = 0.0879956994323094, Validation loss = 0.10336742780378295
Starting Epoch 4 with num_nodes_to_use=5
Epoch 4: Training loss = 0.08473125148323638, Validation loss = 0.10196329640328056
Starting Epoch 5 with num_nodes_to_use=5
Epoch 5: Training loss = 0.07856896258643511, Validation loss = 0.08599737364177902
Starting Epoch 6 with num_nodes_to_use=5
Epoch 6: Training loss = 0.07536093004443258, Validation loss = 0.08761678775772452
Starting Epoch 7 with num_nodes_to_use=5
Epoch 7: Training loss = 0.07315772544487595, Validation loss = 0.08079501308707727
Starting Epoch 8 with num_nodes_to_use=5
Epoch 8: Training loss = 0.07333572252744663, V


KeyboardInterrupt



In [18]:
def train_model_in_stages_with_summed_predictions(model_class, train_loader, test_loader, 
                                                  save_loc, device, loss_fn, 
                                                  num_node_stages=[5, 10, 15], max_epochs=50, 
                                                  early_stopping_patience=50):
    
    
    for stage_idx,num_nodes_to_use in enumerate(num_node_stages):
        
        print(f"Training model with num_nodes_to_use={num_nodes_to_use}")
        
        # Initialize the model from scratch for each stage using Xavier initialization
        model = model_class().to(device)
        optimizer = torch.optim.Adam(model.parameters(),lr=0.00075) 
   
        # Initialize scheduler and early stopping variables
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.75, patience=5, verbose=True)
        
        best_val_loss = float('inf')
        early_stop_counter = 0
        best_model_state = None
        
        epoch_loss_train = []
        epoch_loss_val = []
        lr_list = []
        
        ### loading the frozen models ###
        if num_nodes_to_use==10:
            
            frozen_model_5 = model_class().to(device)
            frozen_model_5.load_state_dict(torch.load( save_loc + "best_model_num_nodes_5.pt"))
            
            for param in frozen_model_5.parameters():
                param.requires_grad = False
            
        if num_nodes_to_use==15:
            
            frozen_model_5 = model_class().to(device)
            frozen_model_5.load_state_dict(torch.load( save_loc + "best_model_num_nodes_5.pt"))
            
            for param in frozen_model_5.parameters():
                param.requires_grad = False
            
            frozen_model_10 = model_class().to(device)
            frozen_model_10.load_state_dict(torch.load( save_loc + "best_model_num_nodes_10.pt"))
            
            for param in frozen_model_10.parameters():
                param.requires_grad = False
        
        for epoch in range(max_epochs):
            print(f'Starting Epoch {epoch+1} with num_nodes_to_use={num_nodes_to_use}')
            
            loss_train = []
            loss_val = []
            
            # Training loop
            model.train()
            for batch, inputs_global in train_loader:
                
                batch.to(device)
                inputs_global = inputs_global.float().cuda()

                optimizer.zero_grad()

                # Get the predictions from the current model
                current_pred = model(batch.x.float(), inputs_global, batch, num_nodes_to_use=num_nodes_to_use)
                
                if num_nodes_to_use == 5:
                    total_pred = current_pred
                    
                if num_nodes_to_use == 10:
                    with torch.no_grad():
                        frozen_pred_5 = frozen_model_5(batch.x.float(), inputs_global, batch, num_nodes_to_use=5)
                    total_pred = current_pred + frozen_pred_5
                    
                if num_nodes_to_use == 15:
                    with torch.no_grad():
                        frozen_pred_5 = frozen_model_5(batch.x.float(), inputs_global, batch, num_nodes_to_use=5)
                        frozen_pred_10 = frozen_model_10(batch.x.float(), inputs_global, batch, num_nodes_to_use=10)
                    total_pred = current_pred + frozen_pred_5 + frozen_pred_10

                # Compute loss based on the summed predictions
                loss = loss_fn(total_pred, batch.y)
                loss.backward()

                optimizer.step()

                loss_train.append(loss.item())
            
            # Validation loop
            model.eval()
            
            with torch.no_grad():
                for batch, inputs_global in test_loader:
                    batch.to(device)
                    inputs_global = inputs_global.float().cuda()

                    # Get predictions from the current model
                    current_pred = model(batch.x.float(), inputs_global, batch, num_nodes_to_use=num_nodes_to_use)

                    if num_nodes_to_use == 5:
                        total_pred = current_pred
                    
                    if num_nodes_to_use == 10:
                        with torch.no_grad():
                            frozen_pred_5 = frozen_model_5(batch.x.float(), inputs_global, batch, num_nodes_to_use=5)
                        total_pred = current_pred + frozen_pred_5
                    
                    if num_nodes_to_use == 15:
                        with torch.no_grad():
                            frozen_pred_5 = frozen_model_5(batch.x.float(), inputs_global, batch, num_nodes_to_use=5)
                            frozen_pred_10 = frozen_model_10(batch.x.float(), inputs_global, batch, num_nodes_to_use=10)
                        total_pred = current_pred + frozen_pred_5 + frozen_pred_10

                    loss = loss_fn(total_pred, batch.y)
                    loss_val.append(loss.item())

            print(f'Epoch {epoch+1}: Training loss = {np.mean(loss_train)}, Validation loss = {np.mean(loss_val)}')
            
            # Store epoch losses
            epoch_loss_train.append(np.mean(loss_train))
            epoch_loss_val.append(np.mean(loss_val))
            
            # Save the best model if validation loss improves
            if epoch_loss_val[-1] < best_val_loss:
                best_val_loss = epoch_loss_val[-1]
                best_model_state = model.state_dict().copy()  # Save best model state
                torch.save(best_model_state, save_loc + f'best_model_num_nodes_{num_nodes_to_use}.pt')
                early_stop_counter = 0
            else:
                early_stop_counter += 1
            
            # Learning rate scheduling
            lr_list.append(optimizer.param_groups[0]['lr'])
            scheduler.step(epoch_loss_val[-1])

            # Early stopping condition
            if early_stop_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1} with num_nodes_to_use={num_nodes_to_use}")
                break
            
            # Save model every 25 epochs
            if (epoch + 1) % 25 == 0:
                torch.save(model.state_dict(), save_loc + f'model_epoch_{epoch+1}_num_nodes_{num_nodes_to_use}.pt')

            # Save loss history to disk
            np.save(save_loc + "epoch_loss_train_"+str(num_nodes_to_use)+"_nodes.npy", epoch_loss_train)
            np.save(save_loc + "epoch_loss_val_"+str(num_nodes_to_use)+"_nodes.npy", epoch_loss_val)
        
        print(f"Training completed for num_nodes_to_use={num_nodes_to_use}. Best validation loss: {best_val_loss}")

    print("Training across all stages has completed.")
    return best_model_state  # Optionally return the best model state from the final stage

In [ ]:
# Train the model #

# Train model in stages with sequential num_nodes_to_use
best_model_state = train_model_in_stages_with_summed_predictions(
    model_class=GCN, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    save_loc="temp_results/", 
    device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu"), 
    loss_fn=torch.nn.HuberLoss(delta=0.25), 
    num_node_stages=[5, 10, 15], 
    max_epochs=75, 
    early_stopping_patience=50   
)

# Optionally load the best model state for further use
# model.load_state_dict(best_model_state)

Training model with num_nodes_to_use=5
Starting Epoch 1 with num_nodes_to_use=5
Epoch 1: Training loss = 0.09261257646411904, Validation loss = 0.0965248519200409
Starting Epoch 2 with num_nodes_to_use=5
Epoch 2: Training loss = 0.08895202428312962, Validation loss = 0.08825849257344559
Starting Epoch 3 with num_nodes_to_use=5
Epoch 3: Training loss = 0.08657634454248704, Validation loss = 0.08571389149776928
Starting Epoch 4 with num_nodes_to_use=5


# Paper Plots

In [ ]:
epoch_loss_train_1 = np.load(location+"results_type_5_trial_1/epoch_loss_train.npy")
epoch_loss_val_1 = np.load(location+"results_type_5_trial_1/epoch_loss_val.npy")

# epoch_loss_train_2 = np.load(location+"results_type_5_trial_2/epoch_loss_train.npy")
# epoch_loss_val_2 = np.load(location+"results_type_5_trial_2/epoch_loss_val.npy")

# epoch_loss_train_3 = np.load(location+"results_type_5_trial_3/epoch_loss_train.npy")
# epoch_loss_val_3 = np.load(location+"results_type_5_trial_3/epoch_loss_val.npy")

plt.semilogy(epoch_loss_val_1[1:],c="red")
# plt.semilogy(epoch_loss_val_2[1:],c="blue")
# plt.semilogy(epoch_loss_val_3[1:],c="green")
print(epoch_loss_val_1[1:].min(),epoch_loss_val_2[1:].min(),epoch_loss_val_3[1:].min())

plt.legend(["Trial 1","Trial 2","Trial 3"])

In [ ]:
### Load the model ###
model.load_state_dict(torch.load( location+ "results_type_5_trial_1/best_model.pt"))
model = model.cuda()

In [ ]:
def get_predictions(inputs_combined,inputs_global,trained_model):
    
    loader = DataLoader(list(zip(inputs_combined,inputs_global)), batch_size=1, shuffle=False)
    df_result = list()
    k=0
    with torch.no_grad():

        for batch,inputs_global in loader:
            print("Particle number : ",str(k+1))
            batch.to(device) 
            inputs_global = inputs_global.float().cuda() 

            # Passing the node features and the connection info
            pred = model( batch.x.float() , batch.edge_index, inputs_global, batch.batch)
            df_result.append(np.array([batch.y.detach().cpu().numpy()[0][0],pred.detach().cpu().numpy()[0][0]]))    
            
            k=k+1
            clear_output(wait=True)
            
    df_result = np.stack(df_result)
    
    return df_result

In [ ]:
def predictions_post_proc(gt_pred_list):
    
    ### Convert to pandas dataframes ###
    gt_pred_list_pd = [ pd.DataFrame(gt_pred_list[i],columns=["GT Drag","Pred Drag"]) for i in range(len(gt_pred_list)) ]

    for i in range(len(gt_pred_list_pd)):
        ### Define Relative error from prediction ###
        gt_pred_list_pd[i]["Rel Err"] = np.abs(gt_pred_list_pd[i]["GT Drag"] - gt_pred_list_pd[i]["Pred Drag"]) / np.abs(gt_pred_list_pd[i]["GT Drag"])
    
        ### Define column of mean gt drag force ###
        gt_pred_list_pd[i]["Mean GT Drag"] = np.ones(len(gt_pred_list_pd[i]))*gt_pred_list_pd[i]["GT Drag"].mean()

        ### Relative error from mean gt drag force ###
        gt_pred_list_pd[i]["Rel Err From Mean Drag GT"] = np.abs(gt_pred_list_pd[i]["Mean GT Drag"] - gt_pred_list_pd[i]["GT Drag"]) / np.abs(gt_pred_list_pd[i]["GT Drag"])
        
    return gt_pred_list_pd

In [ ]:
def plot_drag_scatter(dataframes_list, nrows=3, ncols=3):
    num_dfs = len(dataframes_list)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), squeeze=False)
    
    for idx, df in enumerate(dataframes_list):
        row = idx // ncols
        col = idx % ncols
        ax = axes[row, col]
        
        # Scatter plot
        ax.scatter(df['GT Drag'], df['Pred Drag'], alpha=0.5)
        ax.plot(np.linspace(-0.2,1.2,10) , np.linspace(-0.2,1.2,10),c='black')
        
        # Calculate R²
        r2 = r2_score(df['GT Drag'], df['Pred Drag'])
        
        # Add R² text to the plot
        ax.text(0.05, 0.95, f'R² = {r2:.2f}', transform=ax.transAxes, 
                fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
        
        # Set title
        title = f"Density_ratio: {df['Density_ratio'].iloc[0]}, glb_phi: {df['glb_phi'].iloc[0]}, glb_Re: {df['glb_Re'].iloc[0]}"
        ax.set_title(title, fontsize=10)
        
        # Set labels
        ax.set_xlabel('GT Drag')
        ax.set_ylabel('Pred Drag')
        
        ax.set_xlim([0.2,0.5])
        ax.set_ylim([0.2,0.5])
    
    # Remove empty subplots
    for idx in range(num_dfs, nrows * ncols):
        fig.delaxes(axes.flatten()[idx])
    
    plt.tight_layout()
    plt.show()

In [ ]:
### Do the inference ###
result_test = get_predictions(test_combined,test_input_global,model)
print("R2 score : ",str(r2_score(result_test[:,0:1],result_test[:,1:2])))

In [ ]:
### load the scaler ###
import joblib
scaler = RobustScaler()
scaler = joblib.load(location + "scaler.save") 

In [ ]:
### Convert to pd.Dataframe and post proc (Train)###
# result_train_pd = pd.DataFrame(result_train,columns=["GT Drag","Pred Drag"])
# train_input_global = pd.read_csv(location+"train_input_scalar_unscaled")
# train_input_global = train_input_global.drop('Unnamed: 0', axis=1)
# result_train_pd = pd.concat((train_input_global,result_train_pd),axis=1)

### resacling ###
# result_train_pd["GT Drag"] = result_train_pd["GT Drag"].values*(drag_max-drag_min) + drag_min
# result_train_pd["Pred Drag"] = result_train_pd["Pred Drag"].values*(drag_max-drag_min) + drag_min

### Define Relative error ###
# result_train_pd["Rel Err"] = np.abs(result_train_pd["GT Drag"].values - result_train_pd["Pred Drag"].values)/result_train_pd["GT Drag"].values
# result_train_pd["Rel Err From Mean"] = np.abs(result_train_pd["GT Drag"].values - result_train_pd["GT Drag"].values.mean())/result_train_pd["GT Drag"].values


### Convert to pd.Dataframe and post proc (Test) ###
result_test_pd = pd.DataFrame(result_test,columns=["GT Drag","Pred Drag"])
test_input_global = pd.read_csv(location+"test_input_scalar_unscaled")
test_input_global = test_input_global.drop('Unnamed: 0', axis=1)
result_test_pd = pd.concat((test_input_global,result_test_pd),axis=1)

### rescaling for min max scaling ###
# result_test_pd["GT Drag"] = result_test_pd["GT Drag"].values*(drag_max-drag_min) + drag_min
# result_test_pd["Pred Drag"] = result_test_pd["Pred Drag"].values*(drag_max-drag_min) + drag_min


### rescaling for robust scaling ###
result_test_pd["GT Drag"] = scaler.inverse_transform(result_test_pd["GT Drag"].values[:,None])
result_test_pd["Pred Drag"] = scaler.inverse_transform(result_test_pd["Pred Drag"].values[:,None])

### Define Relative error ###
result_test_pd["Rel Err"] = np.abs(result_test_pd["GT Drag"].values - result_test_pd["Pred Drag"].values)/np.abs(result_test_pd["GT Drag"].values)
result_test_pd["Rel Err From Mean"] = np.abs(result_test_pd["GT Drag"].values - result_test_pd["GT Drag"].values.mean())/np.abs(result_test_pd["GT Drag"].values)

In [ ]:
plt.figure(figsize=(10,5))

plt.scatter(result_test_pd["GT Drag"],result_test_pd["Pred Drag"],alpha=0.9,
            c=np.arange(len(result_test_pd)))
plt.plot(np.linspace(-20,150,10),np.linspace(-20,150,10))

print(r2_score(result_test_pd["GT Drag"],result_test_pd["Pred Drag"]))

plt.xlim([-1,drag_max+5])
plt.ylim([-1,drag_max+5])
print("Max GT Drag from data and inverse transform: ",drag_max)

In [ ]:
result_test_pd["GT Drag"].values.max()

In [ ]:
print("MRP from predictions and drag : ",result_test_pd["Rel Err"].mean(),result_test_pd["Rel Err From Mean"].mean())

In [ ]:
### Save The results ###
result_test_pd.to_csv(location+"/results_type_5_trial_1/results.csv")

# GNN MLP comparision

In [ ]:
location = "../extrapolation/rho2_10percent_Re200/80_20_time_split/"

gnn = pd.read_csv(location+"/results_type_5_trial_1/results.csv")
mlp = pd.read_csv(location+"/results_MLP/results.csv")

print("GNN MRPE",gnn["Rel Err"].values.mean())
print("MLP MRPE",mlp["Rel Err"].values.mean())

no_bins=5000
count_gnn, bins_count_gnn = np.histogram(gnn["Rel Err"].values, bins=no_bins) 
pdf_gnn = count_gnn / sum(count_gnn) 
cdf_gnn = np.cumsum(pdf_gnn) 

count_mlp, bins_count_mlp = np.histogram(mlp["Rel Err"].values, bins=no_bins) 
pdf_mlp = count_mlp / sum(count_mlp) 
cdf_mlp = np.cumsum(pdf_mlp) 

plt.semilogx(bins_count_gnn[1:], cdf_gnn, label="CDF",color="blue",linewidth=1.5) 
plt.semilogx(bins_count_mlp[1:], cdf_mlp, label="CDF",color="red",linewidth=1.5) 

plt.legend(["GNN","MLP"])

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(15,3))

axs[0].scatter(gnn["GT Drag"],gnn["Pred Drag"])
axs[0].plot(np.linspace(-100,100),np.linspace(-100,100))
axs[0].set_xlim([-5,25])
axs[0].set_ylim([-5,25])
axs[0].set_title("GNN R2 = "+str(round(r2_score(gnn["GT Drag"].values,gnn["Pred Drag"].values),4)))

axs[1].scatter(mlp["GT Drag"],mlp["Pred Drag"])
axs[1].plot(np.linspace(-100,100),np.linspace(-100,100))
axs[1].set_xlim([-5,25])
axs[1].set_ylim([-5,25])
axs[1].set_title("MLP R2 = "+str(round(r2_score(mlp["GT Drag"].values,mlp["Pred Drag"].values),4)))

# Neighbor Influence

In [ ]:
neigh_1 = np.load("../extrapolation/rho2_10percent_Re200/80_20_time_split_1_neighbors/results_type_5_trial_1/epoch_loss_val.npy")
neigh_5 = np.load("../extrapolation/rho2_10percent_Re200/80_20_time_split_5_neighbors/results_type_5_trial_1/epoch_loss_val.npy")
neigh_10 = np.load("../extrapolation/rho2_10percent_Re200/80_20_time_split_10_neighbors/results_type_5_trial_1/epoch_loss_val.npy")

plt.semilogy(neigh_1)
plt.semilogy(neigh_5)
plt.semilogy(neigh_10)

plt.legend(["1","5","10"])

# For comparision with the recurrent model 

In [ ]:
### min max for rho2 30percent 300 (-107.666576,109.943932) ###

In [ ]:
# particle_identifier = pd.read_csv("/home/neilashwinraj/gnns/volatile/test_dataset_identifiers_steady")
# idx = np.where( (particle_identifier["case_ID"]==2
#                 )&
#           (particle_identifier["particle_ID"]==36) )

# test_combined_single_particle = [test_combined[idx[0][i]] for i in range(len(idx[0]))]
# test_input_global_single_particle = [test_input_global[idx[0][i]] for i in range(len(idx[0]))]

# single_particle_data_loader = DataLoader(list(zip(test_combined_single_particle,test_input_global_single_particle)),
#                                          batch_size=1, shuffle=False)

# pred = list()
# gt = list()
# model.eval()

# for batch,inputs_global in single_particle_data_loader:

#         batch.to(device)
#         inputs_global = inputs_global.float().cuda() 

#         pred.append(model( batch.x.float() , batch.edge_index, inputs_global, batch.batch).detach().cpu().numpy()[0][0])
#         gt.append(batch.y.detach().cpu().numpy()[0][0])
        
# ### Unscaling ###

# gt = np.array(gt)*(109.943932-(-107.666576))+(-107.666576)
# pred = np.array(pred)*(109.943932-(-107.666576))+(-107.666576)

In [ ]:
particle_identifier

In [ ]:
unique_cases = np.unique(particle_identifier["case_ID"])
unique_particle_ids = np.unique(particle_identifier["particle_ID"])

In [ ]:
np.where((particle_identifier["case_ID"]==unique_case)&(particle_identifier["particle_ID"]==unique_particle_id))

In [ ]:
particle_identifier = pd.read_csv("/home/neilashwinraj/gnns/volatile/test_dataset_identifiers_steady")

results_list_steady = list()

for unique_case in unique_cases:
    
    for unique_particle_id in unique_particle_ids:
        
        print("case and particle ID = ",unique_case,unique_particle_id)
        
        idx = np.where( (particle_identifier["case_ID"]==unique_case
                        )&(particle_identifier["particle_ID"]==unique_particle_id) )
        
        
        test_combined_single_particle = [test_combined[idx[0][i]] for i in range(len(idx[0]))]
        test_input_global_single_particle = [test_input_global[idx[0][i]] for i in range(len(idx[0]))]

        single_particle_data_loader = DataLoader(list(zip(test_combined_single_particle,test_input_global_single_particle)),
                                                 batch_size=1, shuffle=False)

        pred = list()
        gt = list()
        model.eval()

        for batch,inputs_global in single_particle_data_loader:

                batch.to(device)
                inputs_global = inputs_global.float().cuda() 

                pred.append(model( batch.x.float() , batch.edge_index, inputs_global, batch.batch).detach().cpu().numpy()[0][0])
                gt.append(batch.y.detach().cpu().numpy()[0][0])

        ### Unscaling ###

        gt = np.array(gt)*(109.943932-(-107.666576))+(-107.666576)
        pred = np.array(pred)*(109.943932-(-107.666576))+(-107.666576)
        
        dataframe = pd.DataFrame(np.concatenate((gt[:,None].T,pred[:,None].T),axis=0))
        dataframe.columns = ["t = "+str(i+1) for i in range(dataframe.shape[1])]
        dataframe.index = ["GT Drag","Pred Drag"]
        dataframe.loc['relative_error'] = np.abs(dataframe.loc["GT Drag"] - dataframe.loc["Pred Drag"]).values/np.abs(dataframe.loc["GT Drag"])
        
        results_list_steady.append(dataframe)
        
        clear_output(wait=True)

In [ ]:
import pickle
with open('/home/neilashwinraj/gnns/volatile/results_list_steady.pkl', 'wb') as file:
    pickle.dump(results_list_steady, file)

In [ ]:
# np.save("/home/neilashwinraj/gnns/volatile/single_particle_predictions_steady",pred)

In [ ]:
plt.plot(gt)
plt.plot(pred)
plt.legend(["gt","pred"])

# Garbage Below

In [ ]:
# ### Final model ###

# class GCN(torch.nn.Module):
#     def __init__(self,embedding_size=200,batch_size=16,num_nodes=16,num_features=4):
        
#         # Init parent
#         super(GCN, self).__init__()
#         torch.manual_seed(42)
#         self.num_nodes = num_nodes

#         # GCN layers
#         self.initial_conv = GCNConv(num_features, embedding_size)
#         self.conv1 = GCNConv(embedding_size, embedding_size)
#         self.conv2 = GCNConv(embedding_size, embedding_size)
#         self.conv3 = GCNConv(embedding_size, num_features)

#         # Output layer
# #         self.linear_1 = Linear(1024+3,128)
#         self.linear_1 = Linear(num_features+4,128)
#         self.linear_2 = Linear(128,1)

#     def forward(self, x, edge_index, x_scalar, batch_index):

#         # First Conv layer
#         hidden = self.initial_conv(x, edge_index)
#         hidden = F.leaky_relu(hidden)
        
#         # Other Conv layers
#         hidden = self.conv1(hidden, edge_index)
#         hidden = F.leaky_relu(hidden)
        
#         hidden = self.conv2(hidden, edge_index)
#         hidden = F.leaky_relu(hidden)
        
#         hidden = self.conv3(hidden, edge_index)
#         hidden = F.leaky_relu(hidden)
        
#         print("hidden",hidden.shape)
        
#         x_scalar = x_scalar.unsqueeze(1).repeat(1, self.num_nodes, 1)
#         x_scalar = x_scalar.reshape(x_scalar.shape[0]*x_scalar.shape[1],x_scalar.shape[-1])
        
#         hidden = torch.cat((hidden,x_scalar),axis=1)
        
#         out = self.linear_1(hidden)
#         out = F.leaky_relu(out)
#         out = self.linear_2(out)
        
#         drag_predictions = torch.zeros_like(  torch.unique(batch_index) , dtype=torch.float32 )[:,None] 
#         drag_predictions = drag_predictions.scatter_add(0, batch_index[:,None], out)
        
#         return drag_predictions

# model = GCN()
# print(model)
# print("Number of parameters: ", sum(p.numel() for p in model.parameters()))